In [ ]:
# Clone + install OUR package into the main Colab env (training stack).
!git clone https://github.com/sukhrobnurali/tooltuned-qwen.git
%cd tooltuned-qwen
!pip install -e ".[colab]" --quiet
!pip uninstall -y torchcodec --quiet

In [ ]:
# bfcl-eval install on Colab fights five upstream issues stacked on each other:
#   1. bfcl-eval's vLLM pin conflicts with our `[colab]` extras' torch pin --
#      isolate bfcl-eval in its own venv (`run_bfcl(bfcl_executable=...)`
#      shells out to the venv binary).
#   2. bfcl-eval pins vllm==0.8.5, no Python 3.12 wheels -- use Python 3.11
#      via uv.
#   3. bfcl-eval pins `mistralai==1.7.0`, but mistralai is currently
#      QUARANTINED on PyPI (`pypi:project-status=quarantined` on the simple
#      index, all versions hidden). Mistralai's GitHub has the v1.7.0 tag,
#      but poetry's build fails on the tag because `README-PYPI.md` (a file
#      auto-generated for PyPI uploads only) is missing from the git
#      checkout. Workaround: clone mistralai locally, `touch` the missing
#      README, install from the local path, then strip the mistralai pin
#      from bfcl-eval's pyproject so its install doesn't try to re-fetch.
#   4. vllm is in `[project.optional-dependencies] oss_eval_vllm`, not a base
#      dep -- must install with `[oss_eval_vllm]` extra or bfcl's `--backend
#      vllm` flag crashes at `FileNotFoundError: vllm`.
#   5. vLLM 0.8.5 was built against transformers v4 (uses the
#      `all_special_tokens_extended` tokenizer attribute that v5 removed).
#      Nothing in bfcl-eval caps transformers, so a fresh install pulls v5
#      and `vllm serve` crashes at tokenizer init. Pin `transformers<5`
#      explicitly after the bfcl-eval install.
#
# Heavy install: vLLM 0.8.5 is several GB and rebuilds CUDA extensions;
# expect 5-10 minutes on a fresh A100 runtime.
!pip install uv --quiet
!uv python install 3.11
!uv venv /content/bfcl-venv --python 3.11 --clear

# Clone mistralai's v1.7.0 tag locally and patch in the missing README that
# poetry's build backend requires; install into the venv first so bfcl-eval
# sees mistralai already satisfied at install time.
!rm -rf /content/mistralai
!git clone --depth=1 --branch v1.7.0 https://github.com/mistralai/client-python.git /content/mistralai
!touch /content/mistralai/README-PYPI.md
!uv pip install --python /content/bfcl-venv/bin/python /content/mistralai

# Clone gorilla shallowly and STRIP the mistralai dep line from bfcl-eval's
# pyproject (we pre-installed it above).
!rm -rf /content/gorilla
!git clone --depth=1 https://github.com/ShishirPatil/gorilla.git /content/gorilla
!sed -i '/"mistralai==1.7.0"/d' /content/gorilla/berkeley-function-call-leaderboard/pyproject.toml

# Install bfcl-eval WITH the `[oss_eval_vllm]` extra so vllm==0.8.5 lands
# in the venv. Verbose (no --quiet) so any future dep conflict surfaces.
!uv pip install --python /content/bfcl-venv/bin/python "/content/gorilla/berkeley-function-call-leaderboard[oss_eval_vllm]"

# Downgrade transformers to v4 -- vLLM 0.8.5's tokenizer code uses the v4
# attribute `all_special_tokens_extended` that v5 dropped; without this pin
# vllm serve crashes at tokenizer init with AttributeError.
!uv pip install --python /content/bfcl-venv/bin/python "transformers<5"

# qwen_agent transitively imports soundfile at module load even though we
# never touch audio. Tiny install, libsndfile is already on Colab images.
!uv pip install --python /content/bfcl-venv/bin/python soundfile

# Sanity check: bfcl + vllm binaries should both exist in the venv.
!/content/bfcl-venv/bin/bfcl --help | head -5
!ls /content/bfcl-venv/bin/vllm
!/content/bfcl-venv/bin/python -c "import transformers; print('transformers:', transformers.__version__)"

In [ ]:
import sys, os
sys.path.insert(0, "/content/tooltuned-qwen/src")
from google.colab import userdata
for key in ("HF_TOKEN",):
    val = userdata.get(key)
    assert val, f"Set {key} in Colab secrets and toggle Notebook access on"
    os.environ[key] = val
BFCL_BIN = "/content/bfcl-venv/bin/bfcl"
BFCL_PY = "/content/bfcl-venv/bin/python"
print("ready")

In [ ]:
# bfcl-eval has a 2000-line registry of supported models. To run BFCL on
# a model that isn't there yet we inject ModelConfig entries by editing
# bfcl_eval/constants/{model_config,supported_models}.py in the venv.
# The patch script runs via the venv's Python (since `import bfcl_eval`
# fails from this kernel -- bfcl-eval lives in the isolated venv only).
import pathlib, subprocess
patch_script = pathlib.Path("/content/patch_bfcl.py")
patch_script.write_text(r'''
import re, pathlib, bfcl_eval
config_path = pathlib.Path(bfcl_eval.__file__).parent / "constants" / "model_config.py"
supported_path = config_path.parent / "supported_models.py"
src = config_path.read_text(encoding="utf-8")
if "tooltuned-qwen-3.5-4b-FC" in src:
    print("already registered, skipping")
else:
    custom_block = """
# === tooltuned-qwen custom entries (Phase 4 BFCL eval) ===
custom_inference_model_map = {
    \"tooltuned-qwen-3.5-4b-FC\": ModelConfig(
        model_name=\"Qwen/Qwen3.5-4B\",
        display_name=\"tooltuned-qwen-3.5-4b (FC)\",
        url=\"https://huggingface.co/sukhrobnurali/tooltuned-qwen-3.5-4b\",
        org=\"Sukhrob Nurali\",
        license=\"apache-2.0\",
        model_handler=QwenFCHandler,
        input_price=None,
        output_price=None,
        is_fc_model=True,
        underscore_to_dot=False,
    ),
    \"tooltuned-qwen-3.5-4b\": ModelConfig(
        model_name=\"Qwen/Qwen3.5-4B\",
        display_name=\"tooltuned-qwen-3.5-4b (Prompt)\",
        url=\"https://huggingface.co/sukhrobnurali/tooltuned-qwen-3.5-4b\",
        org=\"Sukhrob Nurali\",
        license=\"apache-2.0\",
        model_handler=QwenHandler,
        input_price=None,
        output_price=None,
        is_fc_model=False,
        underscore_to_dot=False,
    ),
    \"qwen3.5-4b-base-FC\": ModelConfig(
        model_name=\"Qwen/Qwen3.5-4B\",
        display_name=\"Qwen3.5-4B (FC, baseline)\",
        url=\"https://huggingface.co/Qwen/Qwen3.5-4B\",
        org=\"Qwen\",
        license=\"apache-2.0\",
        model_handler=QwenFCHandler,
        input_price=None,
        output_price=None,
        is_fc_model=True,
        underscore_to_dot=False,
    ),
    \"qwen3.5-4b-base\": ModelConfig(
        model_name=\"Qwen/Qwen3.5-4B\",
        display_name=\"Qwen3.5-4B (Prompt, baseline)\",
        url=\"https://huggingface.co/Qwen/Qwen3.5-4B\",
        org=\"Qwen\",
        license=\"apache-2.0\",
        model_handler=QwenHandler,
        input_price=None,
        output_price=None,
        is_fc_model=False,
        underscore_to_dot=False,
    ),
}
"""
    src = src.replace(
        "MODEL_CONFIG_MAPPING = {",
        custom_block + "\nMODEL_CONFIG_MAPPING = {",
    )
    src = re.sub(
        r"(MODEL_CONFIG_MAPPING = \{[^}]*?)(\n\})",
        r"\1    **custom_inference_model_map,\2",
        src,
        count=1,
    )
    config_path.write_text(src, encoding="utf-8")
    sup_src = supported_path.read_text(encoding="utf-8")
    if "tooltuned-qwen-3.5-4b-FC" not in sup_src:
        sup_src = sup_src.replace(
            "SUPPORTED_MODELS = [",
            "SUPPORTED_MODELS = [\n    \"tooltuned-qwen-3.5-4b-FC\",\n"
            "    \"tooltuned-qwen-3.5-4b\",\n    \"qwen3.5-4b-base-FC\",\n"
            "    \"qwen3.5-4b-base\",",
        )
        supported_path.write_text(sup_src, encoding="utf-8")
    print(f"registered 4 custom entries in {config_path}")
''')
subprocess.run([BFCL_PY, str(patch_script)], check=True)

In [ ]:
# Pull base weights + LoRA adapter to local /content paths. `--local-model-path`
# wants a directory with config.json + tokenizer + weights, and `--lora-modules`
# wants the adapter dir. Doing this once up front avoids each `bfcl generate`
# re-downloading the base (~8GB) from the HF cache miss.
from huggingface_hub import snapshot_download
base_path = snapshot_download(
    "Qwen/Qwen3.5-4B",
    token=os.environ["HF_TOKEN"],
    local_dir="/content/qwen3.5-4b-base",
)
adapter_path = snapshot_download(
    "sukhrobnurali/tooltuned-qwen-3.5-4b",
    token=os.environ["HF_TOKEN"],
    local_dir="/content/tooltuned-adapter",
)
os.makedirs("/content/bfcl-work", exist_ok=True)
print("base:", base_path)
print("adapter:", adapter_path)

In [ ]:
from tooltuned_qwen.eval.bfcl_runner import run_bfcl
base_fc = run_bfcl(
    model="qwen3.5-4b-base",
    mode="fc",
    out_dir="/content/tooltuned-qwen/results/bfcl/base_fc",
    bfcl_cwd="/content/bfcl-work",
    local_model_path="/content/qwen3.5-4b-base",
    bfcl_executable=BFCL_BIN,
)
print("base FC overall:", base_fc["overall"])

In [ ]:
tuned_fc = run_bfcl(
    model="tooltuned-qwen-3.5-4b",
    mode="fc",
    out_dir="/content/tooltuned-qwen/results/bfcl/tuned_fc",
    bfcl_cwd="/content/bfcl-work",
    local_model_path="/content/qwen3.5-4b-base",
    lora_modules={"tooltuned": "/content/tooltuned-adapter"},
    bfcl_executable=BFCL_BIN,
)
print("tuned FC overall:", tuned_fc["overall"])

In [ ]:
base_prompt = run_bfcl(
    model="qwen3.5-4b-base",
    mode="prompt",
    out_dir="/content/tooltuned-qwen/results/bfcl/base_prompt",
    bfcl_cwd="/content/bfcl-work",
    local_model_path="/content/qwen3.5-4b-base",
    bfcl_executable=BFCL_BIN,
)
print("base Prompt overall:", base_prompt["overall"])

In [ ]:
tuned_prompt = run_bfcl(
    model="tooltuned-qwen-3.5-4b",
    mode="prompt",
    out_dir="/content/tooltuned-qwen/results/bfcl/tuned_prompt",
    bfcl_cwd="/content/bfcl-work",
    local_model_path="/content/qwen3.5-4b-base",
    lora_modules={"tooltuned": "/content/tooltuned-adapter"},
    bfcl_executable=BFCL_BIN,
)
print("tuned Prompt overall:", tuned_prompt["overall"])

In [ ]:
# Build the FC + Prompt comparisons, regenerate the model card with FC numbers
# (per plan 4.1 -- 'model card highlights FC'), and push card + chart to HF.
from tooltuned_qwen.eval.compare import build_comparison
from tooltuned_qwen.hub.model_card import generate_card
from huggingface_hub import HfApi

fc_results = build_comparison(
    base_results=base_fc,
    tuned_results=tuned_fc,
    out_dir="/content/tooltuned-qwen/results/bfcl/fc",
    base_model_name="Qwen/Qwen3.5-4B",
)
prompt_results = build_comparison(
    base_results=base_prompt,
    tuned_results=tuned_prompt,
    out_dir="/content/tooltuned-qwen/results/bfcl/prompt",
    base_model_name="Qwen/Qwen3.5-4B",
    title="BFCL V4 (Prompt mode) -- base vs. fine-tuned Qwen 3.5 4B",
)
print(f"FC     delta: {fc_results['delta']*100:+.2f}pp  (base {fc_results['overall_base']*100:.1f}% -> tuned {fc_results['overall_tuned']*100:.1f}%)")
print(f"Prompt delta: {prompt_results['delta']*100:+.2f}pp  (base {prompt_results['overall_base']*100:.1f}% -> tuned {prompt_results['overall_tuned']*100:.1f}%)")

# Gate check (plan 4.4): >=3pp on FC. If this fires, write ADR 0005 before
# pushing -- don't ship a public artifact below the gate.
if fc_results["delta"] < 0.03:
    print("GATE FAILED: FC delta below 3pp threshold. Write docs/decisions/0005-eval-debugging.md before pushing.")
else:
    card_path = generate_card(
        bfcl_results=fc_results,
        training_config_path="configs/default.yaml",
        out_path="MODEL_CARD.md",
    )
    api = HfApi(token=os.environ["HF_TOKEN"])
    api.upload_file(
        path_or_fileobj=card_path,
        path_in_repo="README.md",
        repo_id="sukhrobnurali/tooltuned-qwen-3.5-4b",
        repo_type="model",
    )
    api.upload_file(
        path_or_fileobj="/content/tooltuned-qwen/results/bfcl/fc/bfcl_comparison.png",
        path_in_repo="bfcl_comparison.png",
        repo_id="sukhrobnurali/tooltuned-qwen-3.5-4b",
        repo_type="model",
    )
    print("model card + chart pushed to HF")